In [65]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression as mutual_info
from sklearn.feature_selection import SequentialFeatureSelector

import warnings
warnings.filterwarnings('ignore')

## Preprocessing
In this first stage, we will prepare the data for analysis. First, we import the datasets:

In [66]:
X_train = pd.read_csv('data/data_student/data_labeled/X_train.csv')
X_test = pd.read_csv('data/data_student/data_labeled/X_test.csv')
y_train = pd.read_csv('data/data_student/data_labeled/y_train.csv')
y_test = pd.read_csv('data/data_student/data_labeled/y_test.csv')
X_train.head(5)

,age,blood_pressure,calcium,cholesterol,hemoglobin,height,potassium,profession,sarsaparilla,smurfberry_liquor,smurfin_donuts,vitamin_D,weight,img_filename
0,161,104.55,2.53,116.47,16.55,7.80,3.98,services,high,very_high,very_low,23.69,113.28,heart_0.png
1,106,99.67,2.43,104.45,14.19,7.32,4.26,services,low,high,low,21.89,85.26,heart_1.png
2,126,119.53,2.61,145.95,15.30,8.01,4.13,craftsmanship,low,moderate,very_high,28.00,122.44,heart_2.png
3,179,122.99,2.22,112.76,16.01,7.81,4.65,food_production,high,very_high,high,21.05,122.17,heart_3.png
4,140,105.63,2.47,112.13,13.92,7.23,3.92,craftsmanship,low,moderate,moderate,36.38,102.84,heart_4.png


#### Missing values
Since all values are non-null, we don't need to handle NaN or null value cases.

In [67]:
X_train.info()
X_test.info()
y_train.info()
y_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                1000 non-null   int64  
 1   blood_pressure     1000 non-null   float64
 2   calcium            1000 non-null   float64
 3   cholesterol        1000 non-null   float64
 4   hemoglobin         1000 non-null   float64
 5   height             1000 non-null   float64
 6   potassium          1000 non-null   float64
 7   profession         1000 non-null   str    
 8   sarsaparilla       1000 non-null   str    
 9   smurfberry_liquor  1000 non-null   str    
 10  smurfin_donuts     1000 non-null   str    
 11  vitamin_D          1000 non-null   float64
 12  weight             1000 non-null   float64
 13  img_filename       1000 non-null   str    
dtypes: float64(8), int64(1), str(5)
memory usage: 109.5 KB
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (to

#### Categorical variables
We can see that some variables are ordinal, specifically **sarsaparilla**, **smurfberry liquor** and **smurfin donuts**. All these variables have possible values: (*very_low* - *low* - *moderate* - *high* - *very_high*). We could assign each value to an integer such that the order is preserved (ordinal encoding), like 1-2-3-4-5 for example.

The **profession** variable doesn't have a specific order, so the best option is to use one-hot encoding. Nevertheless, it seems to be an irrelevant feature so the correct option would be to just drop the column from the dataset. Also, the **img_filename** is not useful to us yet, so we'll drop it as well.

In [68]:
datasets = [X_train, X_test]
columns = ["sarsaparilla", "smurfberry_liquor", "smurfin_donuts"]

ordinal_encoding_map = {"very_low":1, "low":2, "moderate":3, "high":4, "very_high":5}

# Modify columns using ordinal encoding
for dataset in datasets:
    for cloumn in columns:
        dataset[cloumn] = dataset[column].replace(ordinal_encoding_map)

# Drop column "profession" from the datasets
for dataset in datasets:
    del dataset["profession"]
    del dataset["img_filename"]

X_train.head(5)

,age,blood_pressure,calcium,cholesterol,hemoglobin,height,potassium,sarsaparilla,smurfberry_liquor,smurfin_donuts,vitamin_D,weight
0,161,104.55,2.53,116.47,16.55,7.80,3.98,1,1,1,23.69,113.28
1,106,99.67,2.43,104.45,14.19,7.32,4.26,2,2,2,21.89,85.26
2,126,119.53,2.61,145.95,15.30,8.01,4.13,5,5,5,28.00,122.44
3,179,122.99,2.22,112.76,16.01,7.81,4.65,4,4,4,21.05,122.17
4,140,105.63,2.47,112.13,13.92,7.23,3.92,3,3,3,36.38,102.84


#### Outliers
We will check if there are errors in the dataset using percentiles analysis.

In [29]:
(X_train + X_test).describe(percentiles=[0.05,0.95])

,age,blood_pressure,calcium,cholesterol,hemoglobin,height,potassium,vitamin_D,weight
count,500.000000,500.000000,500.00000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000
mean,238.966000,218.011900,4.91626,226.623680,28.554720,15.169660,8.617860,60.396100,212.390400
std,42.065852,16.358225,0.29242,31.991579,2.699614,0.566133,0.709147,7.290912,22.628841
min,97.000000,182.300000,4.22000,121.680000,20.140000,13.610000,6.270000,39.050000,155.960000
5%,171.000000,197.046000,4.44000,171.538000,24.179000,14.269500,7.289500,48.535500,178.670000
95%,308.000000,248.384500,5.39050,278.318000,32.625500,16.150000,9.701000,72.811000,251.786500
max,360.000000,276.620000,5.95000,308.220000,37.610000,16.900000,10.470000,93.110000,297.550000


When comparing the 95th percentile value with the max value for all features, it does not seem like there is any data errors. Therefore, we won't drop any values.

#### Standardization
The next step is to standardize the data

In [69]:
# Standardize predictor variables
scX = StandardScaler()
scX.fit(X_train)
X_train = pd.DataFrame(scX.transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(scX.transform(X_test), columns=X_test.columns)